# Random paths on a cylinder

Which path along the surface of a cylinder (radius $R = 1$) is shortest between two points?
Will Raven's experiment: draw thousands of random wiggly paths, pinned at both ends,
measure each one, and keep the shortest.

A path on the cylinder is $\phi = \phi(z)$ (Taylor Problem 6.2). A short step has
$ds = \sqrt{dz^2 + R^2\,d\phi^2}$, so the length is

$$L = \int_{z_1}^{z_2} \sqrt{1 + R^2\,\phi'(z)^2}\;dz
\;\approx\; \sum_i \sqrt{\Delta z_i^2 + R^2\,\Delta\phi_i^2}.$$

Start: $z = 1$, $\phi = 0$. End: $z = 0.5$, $\phi = \pi/2$.
Half the paths go the short way around; the other half go the long way
(end at $\phi = \pi/2 - 2\pi$, the same point on the cylinder).

Sliders: `n_paths` (how many random paths), `steps` (kinks per path),
`wiggle` (how far the paths stray). Leave `show_answer` off until HW07 is in.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

R = 1.0
z1, phi1 = 1.0, 0.0                  # start
z2, phi2 = 0.5, np.pi / 2            # end, short way
phi2_long = phi2 - 2 * np.pi         # same end, long way

def random_paths(phi_end, n, steps, max_sigma):
    # a smooth path from start to end, plus a random
    # wiggle that is zero at both ends (a Brownian bridge)
    z = np.linspace(z1, z2, steps + 2)
    dz = abs(z[1] - z[0])
    walk = np.cumsum(np.sqrt(dz) * np.random.randn(n, steps + 2), axis=1)
    walk = walk - walk[:, :1]
    frac = (z - z1) / (z2 - z1)
    bridge = walk - frac * walk[:, -1:]
    sigma = np.random.uniform(0, max_sigma, size=(n, 1))
    phi = phi1 + (phi_end - phi1) * frac + sigma * bridge
    return z, phi

def length(z, phi):
    dz = np.diff(z)
    dphi = np.diff(phi, axis=1)
    return np.sum(np.sqrt(dz**2 + (R * dphi)**2), axis=1)

def plot_cylinder(n_paths=10000, steps=20, wiggle=1.0, show_answer=False):
    half = n_paths // 2
    z, phi_s = random_paths(phi2, half, steps, 2 * wiggle)
    z, phi_l = random_paths(phi2_long, half, steps, 4 * wiggle)
    phis = np.vstack([phi_s, phi_l])
    L = length(z, phis)
    best = phis[np.argmin(L)]

    fig = plt.figure(figsize=(11, 4.5))
    ax = fig.add_subplot(1, 2, 1, projection='3d')
    u, v = np.meshgrid(np.linspace(0, 2 * np.pi, 60), np.linspace(0, 1.1, 2))
    ax.plot_surface(R * np.cos(u), R * np.sin(u), v, color='gray', alpha=0.15)
    ax.plot(R * np.cos(best), R * np.sin(best), z, color='orange', lw=3)
    ax.scatter(R * np.cos([phi1, phi2]), R * np.sin([phi1, phi2]), [z1, z2],
               color='red', s=40)
    ax.set_axis_off()
    ax.set_title(f'Shortest of {n_paths} random paths: L = {L.min():.3f}')

    ax2 = fig.add_subplot(1, 2, 2)
    # bins cover every path, so the density counts the
    # long ones too, even though the plot stops at 10
    ax2.hist(L, bins=np.arange(0, L.max() + 0.1, 0.1), density=True)
    ax2.set_xlim(0, 10)
    ax2.set_xlabel('path length'); ax2.set_ylabel('probability density')
    ax2.set_title('Distribution of path lengths')

    if show_answer:
        # the paths with no wiggle at all (HW07 6.7)
        for pe in (phi2, phi2_long):
            zz, pp = random_paths(pe, 1, 98, 0.0)
            ax.plot(R * np.cos(pp[0]), R * np.sin(pp[0]), zz, 'b--')
            ax2.axvline(length(zz, pp)[0], color='b', lw=2)
        ax2.set_title('Distribution of path lengths (blue: no wiggle)')
    plt.show()

interact(plot_cylinder, n_paths=(10, 20000, 10), steps=(5, 100, 5),
         wiggle=(0.1, 3.0, 0.1), show_answer=False);


## Interesting cases

Every slider change draws a fresh set of random paths.

- **Will's run:** the defaults (10000 paths, 20 steps, `wiggle` = 1). Two humps in the histogram,
  the short-way paths and the long-way paths, each with a sharp left edge. Nothing beats the
  short-way edge, about 1.65. Look at the shape of the orange path.
- **Few paths:** `n_paths` = 10. Nudge `wiggle` back and forth a few times: sometimes the shortest
  of ten is near 1.65, sometimes it is near 2. It is never below the edge.
- **Many kinks:** `steps` = 100. The typical path roughly doubles in length and about a third of
  them run off the right of the histogram, but the left edges do not move. What sets them?
- **Tame paths:** `wiggle` = 0.2. Nearly every path piles up at the two edges.
  **Wild paths:** `wiggle` = 3. The humps spread out; the edges are still there.
- **After HW07 is in:** turn on `show_answer` to compare with the paths that have no wiggle at all (blue).
